In [ ]:
import numpy as np
import pandas as pd
import torch
import yaml
import sys

from pathlib import Path

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as tck
from sklearn.manifold import TSNE

# Repository root
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "config.yaml").is_file() and (p / "src").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Configuration and paths
config_path = ROOT / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

data_root = Path(config["data"]["root_path"])

if not data_root.is_absolute():
    data_root = ROOT / data_root

output_dir = ROOT / config["paths"]["output_dir"]
figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

# Run provenance
experiment_seed = config["experiment"]["seed"]
composition_id = 9

# Load saved analysis artifacts
embeddings = torch.load(output_dir / "embeddings.pt", weights_only=False)
node_embeddings = torch.load(output_dir / "node_embeddings.pt", map_location="cpu", weights_only=False)
temperatures = np.loadtxt(output_dir / "metadata.txt")
att_weights = torch.load(output_dir / "attention_weights.pt", weights_only=False)

# Canonical species order
species_labels = ["Ni", "Zn", "Co", "Cu", "Fe"]
species_colors = [
    "teal",
    "black",
    "#959494",
    "blue",
    "magenta",
]
species_palette = dict(zip(species_labels, species_colors))

# Plotting settings
plt.rcParams.update({
    "font.family": "Helvetica Neue",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

### Plot T-SNE (local node)

In [ ]:
# Select specific temperatures for analysis
all_temperatures = list(range(300, 2101, 50))
node_tsne_temperatures = [300, 500, 900, 2100]
node_tsne_colors = {
    0: "teal",      # Ni
    1: "black",     # Zn
    2: "white",     # Co
    3: "blue",      # Cu
    4: "magenta",   # Fe
}

In [ ]:
embedding_arrays = []
atom_types_by_temperature = {}

for temperature in node_tsne_temperatures:
    # Find the correct position in node_embeddings
    embedding_index = all_temperatures.index(temperature)

    embedding = (node_embeddings[embedding_index].detach().cpu().numpy())

    data_path = data_root / "processed_data" / f"data_1_9_{temperature}.pt"
    data = torch.load(data_path, map_location="cpu", weights_only=False,)

    atom_type = np.argmax(data.x.detach().cpu().numpy(), axis=1)

    if embedding.shape[0] != atom_type.shape[0]:
        raise ValueError(
            f"T={temperature}: {embedding.shape[0]} embeddings "
            f"but {atom_type.shape[0]} atom labels."
        )

    embedding_arrays.append(embedding)
    atom_types_by_temperature[temperature] = atom_type

    print(
        f"T={temperature}: node_embeddings[{embedding_index}], "
        f"shape={embedding.shape}, "
        f"atom types={np.unique(atom_type)}"
    )

In [ ]:
joint_embeddings = np.concatenate(embedding_arrays, axis=0,)

temperature_slices = {}
start = 0

for temperature, embedding in zip(node_tsne_temperatures, embedding_arrays):
    stop = start + embedding.shape[0]
    temperature_slices[temperature] = slice(start, stop)
    start = stop

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=25,
    learning_rate=30,
    random_state=42,
)

joint_tsne = tsne.fit_transform(joint_embeddings)

x_min = joint_tsne[:, 0].min()
x_max = joint_tsne[:, 0].max()
y_min = joint_tsne[:, 1].min()
y_max = joint_tsne[:, 1].max()

x_pad = 0.03 * (x_max - x_min)
y_pad = 0.03 * (y_max - y_min)

shared_xlim = (x_min - x_pad, x_max + x_pad,)
shared_ylim = (y_min - y_pad, y_max + y_pad,)

In [ ]:
for temperature in node_tsne_temperatures:
    tsne_embedding = joint_tsne[temperature_slices[temperature]]

    atom_type = atom_types_by_temperature[temperature]

    edge_colors = ["grey" if atom == 2 else "none" for atom in atom_type]

    colors = [node_tsne_colors[int(atom)] for atom in atom_type]

    fig, ax = plt.subplots(figsize=(2, 2))

    ax.scatter(
        tsne_embedding[:, 0],
        tsne_embedding[:, 1],
        c=colors,
        s=3,
        edgecolors=edge_colors,
        linewidths=0.2,
    )

    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()

    output_path = figure_dir / f"joint_tsne_nodes_T{temperature}.svg"

    # fig.savefig(output_path, dpi=500, bbox_inches="tight", pad_inches=0, transparent=True,)

    plt.show()
    plt.close(fig)

Plot the corresponding real-space Monte Carlo configurations

In [ ]:
from src.utils.visualizations import plot_cube_vista

for temperature in node_tsne_temperatures:

    data_path = data_root / "processed_data" / f"data_1_9_{temperature}.pt"
    data = torch.load(data_path, map_location="cpu", weights_only=False)

    output_path = figure_dir / f"atomic_lattice_{temperature}.png"

    plot_cube_vista(
        data,
        node_tsne_colors,
        exclude_type=None,
        output_file=output_path,
        screenshot=True,
    )

### Configuration embedding analysis

 T-SNE projection

In [ ]:
# Fit tsne

configuration_tsne_model = TSNE(n_components=2, perplexity=12, learning_rate=30, random_state=42)
configuration_tsne = configuration_tsne_model.fit_transform(embeddings)

In [ ]:
fig, ax = plt.subplots(figsize=(3.504, 3.204))

scatter = ax.scatter(
    configuration_tsne[:, 0],
    configuration_tsne[:, 1],
    c=temperatures,
    cmap="viridis",
    linewidths=0.5,
    edgecolors="grey",
    s=20,
    alpha=1,
    rasterized=False,
)

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_xticks([])
ax.set_yticks([])

ax.set_frame_on(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

output_path = figure_dir / "tsne_global.svg"

# fig.savefig(
#     output_path,
#     bbox_inches="tight",
#     pad_inches=0,
#     transparent=True,
# )

plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 0.15))

temperature_norm = mpl.colors.Normalize(vmin=300, vmax=2100)

cbar = mpl.colorbar.ColorbarBase(
    ax,
    cmap=plt.get_cmap("viridis"),
    norm=temperature_norm,
    orientation="horizontal",
)

cbar.set_ticks([300, 600, 900, 1200, 1500, 1800, 2100])
cbar.set_label("Temperature (K)", labelpad=3)

output_path = figure_dir / "tsne_map_legend_horizontal.svg"

# fig.savefig(
#     output_path,
#     format="svg",
#     transparent=True,
#     bbox_inches="tight",
#     pad_inches=0.02,
# )

plt.show()
plt.close(fig)

Attention weights analysis (distributions)

In [ ]:
attention_layers = [0, 1]
attention_temperatures = [300, 2100]

In [ ]:
for layer in attention_layers:
    for temperature in attention_temperatures:

        temp_idx = np.where(temperatures == temperature)[0][0]
        weights = (att_weights[layer][temp_idx].detach().cpu().numpy())

        # Load corresponding atomic configuration
        data_path = data_root / "processed_data" / f"data_1_9_{temperature}.pt" 
        data = torch.load(data_path, weights_only=False)

        type_indices = (data.x.argmax(dim=1).cpu().numpy())

        # Separate attention weights by element
        species_att_weights = [weights[type_indices == i] for i in range(5)]

        # Build dataframe for seaborn
        df = pd.DataFrame({
            "Attention weights": np.concatenate(species_att_weights),
            "Element": np.repeat(species_labels, [len(w) for w in species_att_weights])
        })


        fig, ax = plt.subplots(figsize=(2.8, 2), layout="constrained")

        sns.kdeplot(
            data=df,
            x="Attention weights",
            hue="Element",
            bw_adjust=1.5,
            common_norm=False,
            palette=species_palette,
            linewidth=1.5,
            alpha=0.75,
            legend=False,
            ax=ax
        )

        # Labels
        ax.set_xlabel("Attention weights")
        ax.set_ylabel("Density")

        # Axis formatting
        ax.ticklabel_format(style="sci", axis="both", scilimits=(0, 0), useMathText=True)
        ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
        ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
        ax.tick_params(axis="y", which="both", direction="in")
        ax.set_xlim(2.5e-5, 1.4e-4)
        ax.set_ylim(0, 1e5)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Save each figure separately
        output_path = figure_dir / f"kde_layer{layer + 1}_T{temperature}.svg"
        
        # fig.savefig(output_path, transparent=True, bbox_inches=None)

        plt.show()
        plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

handles = [
    Line2D([0], [0], color=species_palette[label], lw=1.5, alpha=0.75)
    for label in species_labels
]

fig = plt.figure(figsize=(3.5, 0.4))
legend = fig.legend(
    handles=handles,
    labels=species_labels,
    loc="center",
    ncol=1,
    frameon=False,
    handlelength=1.5,
    columnspacing=1.0,
)

# Optional: save the legend separately
# fig.savefig(
#     "outputs/figures/kde_legend.svg",
#     transparent=True,
#     bbox_inches="tight",
#     bbox_extra_artists=[legend],
#     pad_inches=0.05,
# )

plt.show()
plt.close(fig)

In [ ]:
from src.utils.visualizations import plot_scalar_map

layers = [0, 1]
selected_temperatures = [300, 2100]

for layer in layers:
    for temperature in selected_temperatures:

        temp_idx = np.where(temperatures == temperature)[0][0]

        weights = (att_weights[layer][temp_idx].detach().cpu().numpy())

        # Save each figure separately
        output_path = figure_dir / f"attmap_layer{layer + 1}_T{temperature}.png"
        plot_scalar_map(data, weights, exclude_type=None, output_file=output_path, screenshot=True)

In [ ]:
# Plot attention heatmap legend

fig, ax = plt.subplots(figsize=(0.5, 3)) #0.85, 2
ax.set_position([0.3, 0.1, 0.1, 0.8]) # left, bottom, width, height

norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
cbar = mpl.colorbar.ColorbarBase(ax, cmap=plt.get_cmap('viridis'), norm=norm, orientation='vertical')
cbar.set_label('')

# Manually add label on the left
ax.text(3, 0.5, "Normalized attention weight", rotation=90, ha='center', va='center', transform=ax.transAxes)

cbar.set_ticks([0.0, 1.0])

# plt.tight_layout()
# output_path = figure_dir / "att_map_legend.svg"
# plt.savefig(output_path, dpi=500, transparent=True)

### The latent-space order parameter (dot product) distance 

In [ ]:
emb_size = embeddings.size(1)

n_temperatures = 37
op_temperatures = temperatures[:n_temperatures]
n_alloy_id = 10

# average embeddings
emb_avg = torch.zeros((n_temperatures, emb_size))
for i in range(n_alloy_id):
    idx_start = i * n_temperatures
    idx_end = (i + 1) * n_temperatures

    emb_avg += embeddings[idx_start:idx_end]
emb_avg /= n_alloy_id

# compute OP
rss = emb_avg[-1]
ord = emb_avg[0]

basis = ord - rss
basis_norm = basis / torch.norm(basis)
latent_space_OP = torch.matmul((emb_avg - rss), basis_norm)

fig, ax = plt.subplots(figsize=(4,3.00))
line = ax.plot(op_temperatures,latent_space_OP,
        marker='s', 
        markersize=3.5, 
        linestyle='-', 
        linewidth=1, 
        color="#0C7BDC")

ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
ax.set_ylim(-.05,4.5)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"Latent-space order parameter, $\langle OP \rangle_{T}$")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

output_path = figure_dir / "latent_OP.svg"
# fig.savefig(output_path, dpi=500, bbox_inches="tight", pad_inches=0, transparent=True)

plt.show()
plt.close(fig)

In [ ]:
temperatures_np = np.asarray(op_temperatures, dtype=float)

order_parameter_np = (
    latent_space_OP
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)

In [ ]:
from src.utils.WC import compute_WC_ensemble

# Warren-Cowley reference curves

wc_ensemble = compute_WC_ensemble(comp_ID=composition_id, data_root=data_root)

wc_colors = [
    "red",
    "#00c8ff",
    "gray",
    "gold",
    "magenta",
    "#00ff22",
    "#ff9603",
    "black",
    "skyblue",
    "teal",
]


fig, ax = plt.subplots(figsize=(3.5, 2.7))
fig.subplots_adjust(
    left=0.15,
    right=0.99,
    bottom=0.15,
    top=0.99
)

for (pair_key, wc_values), color in zip(wc_ensemble.items(), wc_colors):
    ax.plot(
        temperatures_np,
        wc_values,
        marker="o",
        markersize=2.5,
        color=color,
        linestyle="-",
        linewidth=1,
        label=pair_key,
        alpha=1,
    )

# Zero-WC reference line
ax.axhline(y=0, color="grey", linestyle=":", alpha=0.7)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"Warren-Cowley parameter, $\alpha$")
ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
ax.xaxis.set_minor_locator(tck.AutoMinorLocator())

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(ncol=2, edgecolor="grey", shadow=True, fancybox=False,)

output_path = figure_dir / "WCO.svg"

# fig.savefig(
#     output_path,
#     dpi=500,
#     bbox_inches=None,
#     transparent=True
# )

plt.show()
plt.close(fig)

In [ ]:
pearson_r_dict = {}

for pair, wc_values in wc_ensemble.items():
    wc_values = np.asarray(wc_values, dtype=float)

    r = np.corrcoef(wc_values, order_parameter_np)[0, 1]
    pearson_r_dict[pair] = r

print("Pearson r values for each pair:")

for pair, r in pearson_r_dict.items():
    print(f"{pair}: {r:.4f}")

import matplotlib.patches as patches

elements = ["Co", "Cu", "Fe", "Ni", "Zn"]
rows = elements[::-1]

pearson_matrix = pd.DataFrame(np.nan, index=rows, columns=elements)

for pair, r in pearson_r_dict.items():
    element_a, element_b = pair.split("-")
    if elements.index(element_a) <= elements.index(element_b):
        pearson_matrix.loc[element_b, element_a] = r
    else:
        pearson_matrix.loc[element_a, element_b] = r

# Plotting
fig, ax = plt.subplots(figsize=(4, 3))

cmap = plt.get_cmap("coolwarm")
norm = plt.Normalize(vmin=-1, vmax=1)

# Correlation cells
for row_i, row in enumerate(rows):
    for col_i, col in enumerate(elements):

        value = pearson_matrix.loc[row, col]

        if pd.notna(value):
            color = cmap(norm(value))
            rect = patches.Rectangle(
                (col_i, row_i),
                1,
                1,
                facecolor=color,
                edgecolor="gray",
                linewidth=0.6
            )

            ax.add_patch(rect)

            ax.text(
                col_i + 0.5,
                row_i + 0.5,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=(
                    "black"
                    if abs(value) < 0.5
                    else "white"
                )
            )


# Diagonal cells
diag_color = "#eeeeee"

for element in elements:
    row_i = rows.index(element)
    col_i = elements.index(element)

    rect = patches.Rectangle(
        (col_i, row_i),
        1,
        1,
        facecolor=diag_color,
        edgecolor="lightgray",
        linewidth=0.6,
    )

    ax.add_patch(rect)


# Axis formatting
ax.set_xlim(0, len(elements))
ax.set_ylim(len(rows), 0)
ax.set_xticks(np.arange(len(elements)) + 0.5)
ax.set_yticks(np.arange(len(rows)) + 0.5)
ax.set_xticklabels(elements)
ax.set_yticklabels(rows)
ax.tick_params(axis="both",length=0,)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_aspect("equal")
fig.tight_layout()
output_path = figure_dir / "pearson_correlation.svg"
# fig.savefig(
#     output_path,
#     dpi=500,
#     bbox_inches="tight",
#     pad_inches=0,
#     transparent=True,
# )

plt.show()
plt.close(fig)

In [ ]:
# Recheck scatter plot for Pearson r

fig, axes = plt.subplots(2, 5, figsize=(10, 7))

for i, (pair, values) in enumerate(wc_ensemble.items()):
    ax = axes[i // 5, i % 5]
    ax.scatter(latent_space_OP, values, color="blue", s=20, alpha=0.5)
    ax.set_title(pair)
    ax.set_xlabel("Latent-space order parameter")
    ax.set_ylabel("Warren-Cowley parameters")
    ax.set_xlim(-0.5, 4.5)
    ax.axhline(y=0, color="grey", linestyle=":", alpha=0.1)


plt.tight_layout()
output_path = figure_dir / "OP_WC_scatter.svg"
# plt.savefig(output_path, dpi=500, bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
# Endpoint-normalized 300–2100 K latent-space change

OP_low = order_parameter_np[0]
OP_high = order_parameter_np[-1]

op_total_change = OP_low - OP_high

op_endpoint_change = order_parameter_np - OP_high
op_normalized_change = op_endpoint_change / op_total_change


In [ ]:
# Endpoint-normalized 300–2100 K Warren-Cowley change

wc_normalized_change = {}
wc_total_changes = {}

for pair, wc_values in wc_ensemble.items():
    wc_values = np.asarray(wc_values, dtype=float)

    wc_low = wc_values[0]
    wc_high = wc_values[-1]

    total_change = wc_low - wc_high

    wc_total_changes[pair] = abs(total_change)

    if np.isclose(total_change, 0):
        print(f"Skipping normalized {pair}: negligible endpoint change")
        continue

    wc_normalized_change[pair] = ((wc_values - wc_high) / total_change)

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.25, 0.50]

def first_passage_temperatures_on_cooling(temperatures, ordering_fraction, thresholds):
    # Sort from high T -> low T
    order = np.argsort(temperatures)[::-1]

    T_cooling = temperatures[order]
    fraction_cooling = ordering_fraction[order]

    results = {}

    for threshold in thresholds:
        reached = np.where(fraction_cooling >= threshold)[0]

        results[threshold] = (
            T_cooling[reached[0]]
            if len(reached) > 0
            else np.nan
        )

    return results

latent_thresholds = first_passage_temperatures_on_cooling(
    temperatures_np,
    op_normalized_change,
    thresholds
)

wc_thresholds = {
    pair: first_passage_temperatures_on_cooling(
        temperatures_np,
        fraction_ordered,
        thresholds
    )
    for pair, fraction_ordered in wc_normalized_change.items()
}

# Report first sampled OP threshold crossings
for threshold in thresholds:
    T_cross = latent_thresholds[threshold]

    print(
        f"{threshold:.0%} latent-space ordering developed upon cooling: "
        f"first sampled crossing at {T_cross:.0f} K"
    )
print(wc_thresholds) 

In [ ]:
rows = []

rows.append({
    "Quantity": "Latent OP",
    **{f"T{int(threshold * 100)}": latent_thresholds[threshold] for threshold in thresholds}
})

for pair, values in wc_thresholds.items():
    rows.append({
        "Quantity": pair,
        **{f"T{int(threshold * 100)}": values[threshold] for threshold in thresholds}
    })

threshold_df = pd.DataFrame(rows)

threshold_df

In [ ]:
# First derivative: d(OP)/dT
dOP_dT = np.gradient(order_parameter_np, temperatures_np)

# Second derivative: d²(OP)/dT²
d2OP_dT2 = np.gradient(dOP_dT, temperatures_np)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(3.5, 6.5), sharex=True)
fig.subplots_adjust(
    left=0.22,
    right=0.96,
    bottom=0.08,
    top=0.97,
    hspace=0.08,
)
plot_kwargs = {
    "marker": "s",
    "markersize": 2.5,
    "linestyle": "-",
    "linewidth": 1,
    "color": "#0C7BDC",
}

# Raw latent-space order parameter
axes[0].plot(
    temperatures_np,
    order_parameter_np,
    **plot_kwargs,
)
axes[0].set_ylabel(r"Latent-space order parameter, $\langle OP \rangle_{T}$")

# First derivative
axes[1].plot(
    temperatures_np,
    dOP_dT,
    **plot_kwargs,
)
axes[1].set_ylabel(r"$d\langle OP \rangle_{T}/dT$")

# Second derivative
axes[2].plot(
    temperatures_np,
    d2OP_dT2,
    **plot_kwargs,
)
axes[2].set_ylabel(r"$d^2\langle OP \rangle_{T}/dT^2$")
axes[2].set_xlabel("Temperature (K)")

# Axis formatting
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--",zorder=0)
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.tick_params(which="both", direction="in")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

output_path = figure_dir / "Derivatives_OP.svg"
# plt.savefig(output_path, dpi=500, bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
# Change over each sampled 50-K interval
delta_T = np.diff(temperatures_np)
delta_OP = np.diff(order_parameter_np)

interval_slopes = delta_OP / delta_T
interval_midpoints = (temperatures_np[:-1] + temperatures_np[1:]) / 2

# Monotonicity diagnostic
n_decreasing = np.sum(delta_OP < 0)
n_increasing = np.sum(delta_OP > 0)

print(f"Decreasing intervals: {n_decreasing}/{len(delta_OP)}")
print(f"Increasing intervals: {n_increasing}/{len(delta_OP)}")

steepest_interval_idx = np.argmin(interval_slopes)

print("Steepest sampled interval: "f"{temperatures_np[steepest_interval_idx]:.0f}–"f"{temperatures_np[steepest_interval_idx + 1]:.0f} K")
print("Slope:",interval_slopes[steepest_interval_idx])

In [ ]:
for pair, values in wc_ensemble.items():
    
    # First derivative: d(WC)/dT
    dWC_dT = np.gradient(values, temperatures_np)

    # Second derivative: d²(WC)/dT²
    d2WC_dT2 = np.gradient(dWC_dT, temperatures_np)

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(wc_ensemble)))

for (pair, values), color in zip(wc_ensemble.items(), colors):
    values = np.asarray(values)

    # Derivatives with respect to temperature
    dWC_dT = np.gradient(values, temperatures_np)
    d2WC_dT2 = np.gradient(dWC_dT, temperatures_np)

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(3.5, 3),
        sharex=True,
        layout="constrained",
    )

    # Order parameter
    axes[0].plot(
        temperatures_np,
        values,
        marker="s",
        markersize=3.5,
        linewidth=1,
        color=color,
    )
    axes[0].set_ylabel("WC parameter")

    # First derivative
    axes[1].plot(
        temperatures_np,
        dWC_dT,
        marker="s",
        markersize=3.5,
        linewidth=1,
        color=color,
    )
    axes[1].axhline(
        0,
        color="black",
        linewidth=0.8,
        linestyle="--",
    )
    axes[1].set_ylabel(r"$dWC/dT$")

    # Second derivative
    axes[2].plot(
        temperatures_np,
        d2WC_dT2,
        marker="s",
        markersize=3.5,
        linewidth=1,
        color=color,
    )
    axes[2].axhline(
        0,
        color="black",
        linewidth=0.8,
        linestyle="--",
    )
    axes[2].set_ylabel(r"$d^2WC/dT^2$")
    axes[2].set_xlabel("Temperature (K)")

    # Use the atomic pair as the figure title
    pair_label = "–".join(pair) if isinstance(pair, tuple) else str(pair)
    fig.suptitle(pair_label)

    for ax in axes:
        ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
        ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
        ax.tick_params(which="both", direction="in")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.show()
    plt.close(fig)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(3.5, 6.5), sharex=True)
fig.subplots_adjust(
    left=0.22,
    right=0.96,
    bottom=0.08,
    top=0.97,
    hspace=0.08,
)

colors = plt.cm.tab10(np.linspace(0, 1, len(wc_ensemble)))

for (pair, values), color in zip(wc_ensemble.items(), colors):
    values = np.asarray(values)

    # Derivatives with respect to temperature
    dWC_dT = np.gradient(values, temperatures_np)
    d2WC_dT2 = np.gradient(dWC_dT, temperatures_np)

    label = str(pair)

    axes[0].plot(
        temperatures_np,
        values,
        marker="o",
        markersize=2,
        linewidth=1,
        color=color,
        label=label,
    )

    axes[1].plot(
        temperatures_np,
        dWC_dT,
        marker="o",
        markersize=2,
        linewidth=1,
        color=color,
    )

    axes[2].plot(
        temperatures_np,
        d2WC_dT2,
        marker="o",
        markersize=2,
        linewidth=1,
        color=color,
    )

# Labels
axes[0].set_ylabel("WC parameters")
axes[1].set_ylabel(r"$dWC/dT$")
axes[2].set_ylabel(r"$d^2WC/dT^2$")
axes[2].set_xlabel("Temperature (K)")

# Zero-reference lines
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")

# Axis formatting
for ax in axes:
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.tick_params(which="both", direction="in")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# One shared legend
handles, legend_labels = axes[0].get_legend_handles_labels()

axes[0].legend(
    loc="upper right",
    ncol=2,
    frameon=False,
    fontsize=7,
    handlelength=1.4,
    columnspacing=0.9,
    labelspacing=0.3,
)

output_path = figure_dir / "Derivatives_WC.svg"
# plt.savefig(output_path, dpi=500, bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()
plt.close(fig)